# Module 1, Activity 1: Getting Started with Bedrock and Basic Prompting

In this notebook we are going to demonstrate how to use the `boto3` Python SDK along with a variety of abstractions available through the LangChain package to work with the Amazon Bedrock Foundational Models.  By the end of this notebook you will be able to create a basic chain with a simple prompt capable of asking questions to the model and outputing the answer in a human-readable format.

In [ ]:
import boto3
from langchain_aws import BedrockLLM, ChatBedrock
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
session = boto3.session.Session()
region = session.region_name

## Create Bedrock management connection

The Bedrock client is used as the control for Bedrock.  It can do things like list models, check availability, and manage configurations.  But it doesn't actually do anything with the models.

In [ ]:
bedrock = boto3.client("bedrock", region_name=region)
model_ids = [model["modelId"] for model in bedrock.list_foundation_models()["modelSummaries"]]
print("Available Models:", model_ids)

## Initializing BedrockLLM and invoking a model

Here, the BedrockLLM class from the langchain_aws package is instantiated.  This class serves as a high-level wrapper to interface with AWS-hosted LLMs.
The initialization parameters include the model ID (in this case, "amazon.titan-tg1-large"), region, and the necessary AWS credentials.  Once the instance is created, the invoke method is used to send a prompt ("What is the recipe of mayonnaise?") to the model.  This section demonstrates the fundamental workflow: setting up the model wrapper and making a basic invocation call to test the model’s response, providing a concrete example of how to interact with AWS-hosted generative AI models using LangChain.

Try several different prompts here to see what different types of answers you can get!

In [ ]:
simple_llm = BedrockLLM(
    model_id="amazon.titan-tg1-large",
    region_name=region,
)
print(simple_llm.invoke("What is the recipe for mayonnaise?"))

## Introducing ChatBedrock

BedrockLLM is designed for single-turn, prompt-based interactions where you provide the prompt ("What is the recipe for mayonnaise?") and the model generates an output in one go.  This is fine for simple things, but when you need to have more sophisticated interactions you want something that supports chat-like exchanges where the model can manage context over several turns of dialogue.  Additionally, not all of the available models, including more sophisticated models like Anthropic's Claude 3 Sonnet below, are supported by BedrockLLM.  Hence, we have the more sophisticated ChatBedrock, as shown below.

Also note that the output of ChatBedrock contains much more information than just a text output.

In [ ]:
chat_llm = ChatBedrock(
    model_id="anthropic.claude-3-sonnet-20240229-v1:0",
    region_name=region,
)
print(chat_llm.invoke("What is the recipe for mayonnaise?"))

## Temperature

Temperature is the thing that gives models creativity.  It controls the randomness of the model's responses.  Setting it to 0.0 (the minimum) typically results in a more deterministic and consistent output while setting it to 1.0 (the maximum) results in more creative responses.  

- Low temperature (e.g., 0.0-0.3): The model plays it safe, sticking to the most likely answers. This is great when you want accuracy and consistency, like coding help or fact-based answers.

- High temperature (e.g., 0.7-1.0): The model gets more adventurous, picking less common words and generating more diverse responses. This is useful for creative writing, brainstorming, or when you want unique outputs.

If you set it to 0, the model is basically deterministic—it’ll always give the same answer if asked the same thing.  As you increase temperature you will get more creative (and perhaps unpredictable!) responses.  So let's create a simple prompt and run it a few times.  Experiment with both the prompt and the temperature of the following cell and observe the effects.

In [ ]:
chat_llm_temp = ChatBedrock(
    model_id="anthropic.claude-3-sonnet-20240229-v1:0",
    region_name=region,
    temperature=0.9
)
print(chat_llm_temp.invoke("Write a short bedtime story about a robot."))

## Limiting the number of tokens returned

The cost of using an LLM is dependent on how many tokens are sent back and forth with the model.  The `max_tokens` parameter can provide a limit on how many total tokens are returned.  Limiting the token count can be useful when you need to ensure that the responses remain concise or when working within strict output size constraints.  Experiment with a few different values for this to see how the output changes.

In [ ]:
chat_llm_limited = ChatBedrock(
    model_id="anthropic.claude-3-sonnet-20240229-v1:0",
    region_name=region,
    temperature=0.0,
    max_tokens=10
)
print(chat_llm_limited.invoke("What is the recipe for mayonnaise?"))

## Creating a Chain with Prompts and Output Parsing

In order to turn this into af fully-functioning question-answering system, we need to combine the following three concepts:

### Prompts and Prompt Templates

In LangChain, **system prompts** (provide the foundational instructions and context for the AI model.  They define the model's role, behavior, and any specific output format or constraints, ensuring that all subsequent responses align with the intended purpose.  In contrast, **human prompts** represent the dynamic, user-supplied inputs that drive the conversation or query.  They capture the specific question or instruction that the user wants the model to address.  The `ChatPromptTemplate` acts as a framework that seamlessly integrates both types of prompts into a coherent message sequence.  By combining system and human messages, the `ChatPromptTemplate` creates a structured dialogue that guides the model to generate responses that are both contextually relevant and properly formatted.

### Chains

Chains in LangChain are modular sequences that connect various components—such as prompt templates, language models, and output parsers—into a unified workflow.  They enable developers to construct complex, multi-step processing pipelines where each step transforms or utilizes the output from the previous one.  LCEL (LangChain Chain Expression Language) implements this concept using a concise, pipe operator (|) syntax that clearly defines the data flow between these components.  By leveraging LCEL, you can declaratively compose chains that are both flexible and readable, allowing for rapid prototyping and iterative development of sophisticated AI-powered applications.

### Output Parsers

The `AIMessage` format might not be the best way for the output of your LLM calll to be rendered.  So we can keep adding to the chain by adding another component: `StrOutputParser`.  It converts a language model's output into a clean, plain text string by stripping away any additional metadata or formatting.  So we can see in this example that we just add it modularly to the end of our chain.  There are several other output parsers that you might find helpful.  You can find a listing of them in [this LangChain documentation](https://python.langchain.com/docs/how_to/#output-parsers). 

In this code block we are creating a simple, three-part chain with our prompt, our LLM, and our output parser.

And be sure to check out the system prompt to understand the response you are about to get!  We are not going to tinker too much with prompts just yet.  That will be the subject of the next activity.  Just for now know that you can add some basic instructions here to tell the model what you are looking for.


In [ ]:
system_prompt = SystemMessagePromptTemplate.from_template(
    "You are a helpful assistant. Return your answer in pirate speak."
)
human_prompt = HumanMessagePromptTemplate.from_template("{input}")
prompt = ChatPromptTemplate.from_messages([system_prompt, human_prompt])

# Chain with prompt and LLM
chain = prompt | chat_llm | StrOutputParser()
print(chain.invoke({"input": "What is the recipe for mayonnaise?"}))